# 1. Setup and Data Preparation
We normalize the MNIST images and prepare the labels for the conditional training.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

# Load MNIST
(train_images, train_labels), (_, _) = tf.keras.datasets.mnist.load_data()
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5 # Normalize to [-1, 1]

num_classes = 10
BATCH_SIZE = 256

# Create dataset and include labels
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.shuffle(60000).batch(BATCH_SIZE)


# 2. Conditional Architecture
In a cGAN, we use an Embedding layer to turn the class integer (0-9) into a vector, which is then concatenated with the noise (for the Generator) or the image features (for the Discriminator).


In [ ]:
def make_generator_model():
    # Label input
    label = layers.Input(shape=(1,))
    label_embedding = layers.Embedding(num_classes, 50)(label)
    label_embedding = layers.Dense(7*7*1)(label_embedding)
    label_embedding = layers.Reshape((7, 7, 1))(label_embedding)

    # Noise input
    noise = layers.Input(shape=(100,))
    gen = layers.Dense(7*7*256, use_bias=False)(noise)
    gen = layers.BatchNormalization()(gen)
    gen = layers.LeakyReLU()(gen)
    gen = layers.Reshape((7, 7, 256))(gen)

    # Merge label and noise
    concat = layers.Concatenate()([gen, label_embedding])

    x = layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(concat)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    out = layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x)

    return tf.keras.Model([noise, label], out)

def make_discriminator_model():
    img = layers.Input(shape=(28, 28, 1))
    label = layers.Input(shape=(1,))

    label_embedding = layers.Embedding(num_classes, 50)(label)
    label_embedding = layers.Dense(28*28*1)(label_embedding)
    label_embedding = layers.Reshape((28, 28, 1))(label_embedding)

    # Concat image and label
    concat = layers.Concatenate()([img, label_embedding])

    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same')(concat)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    out = layers.Dense(1)(x)

    return tf.keras.Model([img, label], out)

generator = make_generator_model()
discriminator = make_discriminator_model()

# 3. Training Logic
The loss functions remain the same as a standard GAN, but we pass both the image and its corresponding label during the forward pass.


In [ ]:
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
gen_optimizer = tf.keras.optimizers.Adam(1e-4)
disc_optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images, labels):
    noise = tf.random.normal([BATCH_SIZE, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([noise, labels], training=True)

        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = binary_cross_entropy(tf.ones_like(fake_output), fake_output)

        real_loss = binary_cross_entropy(tf.ones_like(real_output), real_output)
        fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
        disc_loss = real_loss + fake_loss

    generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

# Training loop
for epoch in range(20): # Increased epochs lead to better quality
    for image_batch, label_batch in train_dataset:
        train_step(image_batch, label_batch)
    print(f"Epoch {epoch+1} finished")


# 4. Interactive Generation
Run this cell to generate a specific digit of your choice!


In [ ]:
# --- USER INPUT SECTION ---
digit_to_generate = int(input("Enter a digit (0-9) you want the GAN to generate: "))

if 0 <= digit_to_generate <= 9:
    noise = tf.random.normal([1, 100])
    label = np.array([digit_to_generate])

    generated_img = generator([noise, label], training=False)

    plt.imshow(generated_img[0, :, :, 0] * 127.5 + 127.5, cmap='gray')
    plt.title(f"GAN Generated: {digit_to_generate}")
    plt.axis('off')
    plt.show()
else:
    print("Please enter a valid digit between 0 and 9.")